# EmpathRAG — RoBERTa Emotion Classifier (Day 6)
**MSML641 · NLP Course Project**  
Fine-tune RoBERTa-base + LoRA on GoEmotions → 5-class emotion taxonomy  
Target: Weighted F1 > 0.55 on 5-class test set

---
**Before running:** Runtime → Change runtime type → **A100** → Save  
Cell 1 will hard-stop if the wrong GPU is attached.


In [ ]:
# ── STEP 1: Verify A100 is attached ──────────────────────────────────────────
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print(f"GPU     : {gpu_name}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {vram_gb:.1f} GB")
    print(f"PyTorch : {torch.__version__}")

assert torch.cuda.is_available(), "No GPU found — switch to A100 runtime before proceeding."
print("\n✅ GPU check passed.")


In [ ]:
# ── STEP 2: Install missing packages ─────────────────────────────────────────
# Strategy: Colab already has numpy 2.x, scikit-learn 1.6+, transformers 4.57+,
# torch 2.8+, datasets 2.x. We install ONLY what's missing.
# DO NOT pin numpy or scikit-learn — that caused the previous conflicts.

!pip install -q --upgrade \
    peft>=0.18.0 \
    evaluate==0.4.6 \
    accelerate>=1.0.0

# Verify no numpy conflict
import numpy as np
import sklearn
print(f"numpy       : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

import transformers, peft, evaluate as ev, accelerate
print(f"transformers: {transformers.__version__}")
print(f"peft        : {peft.__version__}")
print(f"evaluate    : {ev.__version__}")
print(f"accelerate  : {accelerate.__version__}")
print("\n✅ All packages ready.")


In [ ]:
# ── STEP 3: Mount Drive, set checkpoint dir, detect existing checkpoints ──────
from google.colab import drive
import os, glob

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/empathrag/emotion_classifier"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoint dir: {SAVE_DIR}")

existing = sorted(glob.glob(os.path.join(SAVE_DIR, "checkpoint-*")))
if existing:
    print(f"\nFound {len(existing)} checkpoint(s):")
    for c in existing:
        print(f"  {c}")
    RESUME_FROM = existing[-1]
    print(f"\nWill resume from: {RESUME_FROM}")
else:
    RESUME_FROM = None
    print("\nNo existing checkpoints — starting fresh.")


In [ ]:
# ── STEP 4: Load GoEmotions dataset, remap 27 → 5 classes ────────────────────
from datasets import load_dataset
from collections import Counter

# 5-class taxonomy:
#   0 = distress   (grief, remorse, fear, sadness)
#   1 = anxiety    (nervousness, confusion, embarrassment)
#   2 = frustration(anger, annoyance, disappointment, disgust)
#   3 = neutral    (neutral)
#   4 = hopeful    (optimism, relief, gratitude, joy, love, admiration,
#                   amusement, approval, caring, curiosity, desire,
#                   excitement, pride, realization, surprise)

LABEL_MAP = {
    "grief": 0, "remorse": 0, "fear": 0, "sadness": 0,
    "nervousness": 1, "confusion": 1, "embarrassment": 1,
    "anger": 2, "annoyance": 2, "disappointment": 2, "disgust": 2,
    "neutral": 3,
    "optimism": 4, "relief": 4, "gratitude": 4, "joy": 4,
    "love": 4, "admiration": 4, "amusement": 4, "approval": 4,
    "caring": 4, "curiosity": 4, "desire": 4, "excitement": 4,
    "pride": 4, "realization": 4, "surprise": 4,
}
LABEL_NAMES = ["distress", "anxiety", "frustration", "neutral", "hopeful"]

print("Loading GoEmotions (simplified split)...")
raw = load_dataset("google-research-datasets/go_emotions", "simplified")
print(f"Train: {len(raw['train']):,} | Val: {len(raw['validation']):,} | Test: {len(raw['test']):,}")

# Inspect label format — GoEmotions simplified stores labels as list of ints
# that are indices into feature_names
feature_names = raw["train"].features["labels"].feature.names
print(f"\nDataset has {len(feature_names)} emotion classes.")
print(f"Sample row labels: {raw['train'][0]['labels']} → "
      f"{[feature_names[i] for i in raw['train'][0]['labels']]}")

def remap(example):
    """Collapse multi-hot 27-class labels to single 5-class coarse label.
    Takes the first label that appears in LABEL_MAP; falls back to neutral (3).
    """
    for lid in example["labels"]:
        name = feature_names[lid]
        if name in LABEL_MAP:
            return {"label": LABEL_MAP[name]}
    return {"label": 3}  # neutral fallback

print("\nRemapping labels...")
dataset = raw.map(remap, desc="27→5 remap")

# Distribution check
dist = Counter(dataset["train"]["label"])
print("\nTrain class distribution:")
for i, name in enumerate(LABEL_NAMES):
    pct = 100 * dist[i] / len(dataset["train"])
    print(f"  {i} {name:<15} {dist[i]:>6,}  ({pct:.1f}%)")


In [ ]:
# ── STEP 5: Tokenize ─────────────────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )

print("Tokenizing (this takes ~2 min)...")
tokenized = dataset.map(tokenize, batched=True, desc="Tokenizing")
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"\nTokenization complete.")
print(f"  Train : {len(tokenized['train']):,}")
print(f"  Val   : {len(tokenized['validation']):,}")
print(f"  Test  : {len(tokenized['test']):,}")


In [ ]:
# ── STEP 6: RoBERTa + LoRA model ─────────────────────────────────────────────
import torch
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

id2label = {i: n for i, n in enumerate(LABEL_NAMES)}
label2id = {n: i for i, n in enumerate(LABEL_NAMES)}

# LoRA config: targets query + value projections in RoBERTa attention
# r=16 → ~3M trainable params vs ~125M full fine-tune
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nTraining on: {device}")


In [ ]:
# ── STEP 7: Evaluation metrics ───────────────────────────────────────────────
import evaluate
import numpy as np

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Weighted F1 — primary metric (handles class imbalance)
    weighted_f1 = f1_metric.compute(
        predictions=preds, references=labels, average="weighted"
    )["f1"]

    # Per-class F1 — required for paper's breakdown table
    per_class = f1_metric.compute(
        predictions=preds, references=labels,
        average=None, labels=list(range(5))
    )["f1"]

    result = {"f1_weighted": weighted_f1}
    for i, name in enumerate(LABEL_NAMES):
        result[f"f1_{name}"] = float(per_class[i])
    return result

print("Metrics function ready (weighted F1 + per-class F1 for all 5 classes).")


In [ ]:
# ── STEP 8: Training configuration ───────────────────────────────────────────
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=SAVE_DIR,

    # Schedule
    num_train_epochs=5,
    per_device_train_batch_size=64,    # A100 40GB handles this fine at max_len=128
    per_device_eval_batch_size=128,
    learning_rate=2e-4,                # Higher than full fine-tune — correct for LoRA
    warmup_ratio=0.1,
    weight_decay=0.01,

    # Checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=3,                # Keep last 3 to save Drive space

    # Speed
    fp16=True,
    dataloader_num_workers=4,

    # Logging
    logging_dir=f"{SAVE_DIR}/logs",
    logging_steps=200,
    report_to="none",
)

print("TrainingArguments set:")
print(f"  Epochs        : {training_args.num_train_epochs}")
print(f"  Train batch   : {training_args.per_device_train_batch_size}")
print(f"  Learning rate : {training_args.learning_rate}")
print(f"  Best metric   : {training_args.metric_for_best_model}")
print(f"  Save dir      : {SAVE_DIR}")
print(f"  Resume from   : {RESUME_FROM}")


In [ ]:
# ── STEP 9: Train ────────────────────────────────────────────────────────────
# Expected time on A100: ~75–90 minutes
# Checkpoints saved to Drive after every epoch — safe to disconnect and resume.
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

print("Starting training...")
print(f"Resuming from: {RESUME_FROM}" if RESUME_FROM else "Fresh training run.")
print("-" * 60)

trainer.train(resume_from_checkpoint=RESUME_FROM)

print("-" * 60)
print("Training complete.")


In [ ]:
# ── STEP 10: Save final model + tokenizer to Drive ───────────────────────────
import os

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")
print("\nContents:")
for f in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, f)
    size  = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
    label = f"{size/1e6:.1f} MB" if size > 0 else "dir"
    print(f"  {f:<45} {label}")


In [ ]:
# ── STEP 11: Evaluate on held-out test set ───────────────────────────────────
import json
import numpy as np

print("Evaluating on test set (5,427 examples)...")
test_results = trainer.evaluate(tokenized["test"])

print("\n=== TEST SET RESULTS ===")
weighted_f1 = test_results["eval_f1_weighted"]
print(f"Weighted F1 : {weighted_f1:.4f}  (target: > 0.55)")
print()
print("Per-class F1:")
for name in LABEL_NAMES:
    val = test_results.get(f"eval_f1_{name}", float("nan"))
    print(f"  {name:<15} {val:.4f}")

print()
if weighted_f1 >= 0.55:
    print("✅ PASS — weighted F1 exceeds target of 0.55")
else:
    print("⚠️  BELOW TARGET")
    print("   Fallback option: lower lr to 1e-4, increase epochs to 8, re-run from checkpoint.")

# Save to Drive for the paper
results_path = f"{SAVE_DIR}/test_results.json"
with open(results_path, "w") as f:
    json.dump({k: float(v) for k, v in test_results.items()}, f, indent=2)
print(f"\nResults saved: {results_path}")


In [ ]:
# ── STEP 12: Confusion matrix + classification report ────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

print("Computing confusion matrix on test set...")
pred_output = trainer.predict(tokenized["test"])
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels, preds, target_names=LABEL_NAMES))

cm = confusion_matrix(labels, preds)
print("=== CONFUSION MATRIX (rows=true, cols=pred) ===")
header = f"{'':>15}" + "".join(f"{n:>13}" for n in LABEL_NAMES)
print(header)
for i, row in enumerate(cm):
    print(f"{LABEL_NAMES[i]:>15}" + "".join(f"{v:>13}" for v in row))

np.save(f"{SAVE_DIR}/confusion_matrix.npy", cm)
print(f"\nConfusion matrix saved to Drive.")


## ✅ Training Complete — Day 10 Checklist

### 1. Download checkpoint from Drive
Navigate in Google Drive to:  
`My Drive/empathrag/emotion_classifier/`

Download the entire folder and place it at:  
`models/emotion_classifier/` in your local repo.

Required files:
- `adapter_config.json`
- `adapter_model.safetensors` (or `.bin`) — ~12 MB LoRA weights
- `config.json`
- `tokenizer.json`, `tokenizer_config.json`, `vocab.json`, `merges.txt`
- `test_results.json` — paste weighted F1 into paper
- `confusion_matrix.npy` — per-class breakdown for paper table

### 2. Run corpus annotation (Day 10, locally)
```bash
python src/models/annotate_corpus.py
```
Updates all 1,674,369 SQLite rows with predicted emotion labels (~45 min on CPU).

### 3. Verify the checkpoint loads locally
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

tok = AutoTokenizer.from_pretrained("models/emotion_classifier")
base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)
model = PeftModel.from_pretrained(base, "models/emotion_classifier").eval()
print("✅ Checkpoint loads correctly.")
```
